## Overview


---------


## Table of Contents

- Overview
- Setup
- Dataset
- Load the LLM
- Interact with your data
- Plot your data with natural language
- Further analysis

----


## Setup


Install the required libraries (skip any that are already available):

In [ ]:
%%capture
!pip install langchain
!pip install langchain-experimental
!pip install langchain-groq
!pip install python-dotenv
!pip install matplotlib
!pip install seaborn

### Importing required libraries

_It is recommended that you import all required libraries in one place (here):_


In [ ]:
# Suppress noisy warnings.
import warnings
warnings.filterwarnings('ignore')

from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent

import matplotlib.pyplot as plt
import pandas as pd

## Dataset


The dataset you are using is for the mathematics course. The number of mathematics students involved in the collection was 395. The data collected in locations such as Gabriel Pereira and Mousinho da Silveira includes several pertinent values. Examples of such data are records of demographic information, grades, and alcohol consumption.


| Field     | Description                                                                 |
|-----------|-----------------------------------------------------------------------------|
| school    | GP/MS for the student's school                                              |
| sex       | M/F for gender                                                              |
| age       | 15-22 for the student's age                                                 |
| address   | U/R for urban or rural, respectively                                        |
| famsize   | LE3/GT3 for less than or greater than three family members                  |
| Pstatus   | T/A for living together or apart from parents, respectively                 |
| Medu      | 0 (none) / 1 (primary-4th grade) / 2 (5th - 9th grade) / 3 (secondary) / 4 (higher) for mother's education |
| Fedu      | 0 (none) / 1 (primary-4th grade) / 2 (5th - 9th grade) / 3 (secondary) / 4 (higher) for father's education |
| Mjob      | 'teacher,' 'health' care related, civil 'services,' 'at_home' or 'other' for the student's mother's job |
| Fjob      | 'teacher,' 'health' care related, civil 'services,' 'at_home' or 'other' for the student's father's job |
| reason    | reason to choose this school (nominal: close to 'home', school 'reputation', 'course' preference or 'other') |
| guardian  | mother/father/other as the student's guardian                               |
| traveltime| 1 (<15mins) / 2 (15 - 30 mins) / 3 (30 mins - 1 hr) / 4 (>1hr) for a time from home to school |
| studytime | 1 (<2hrs) / 2 (2 - 5hrs) / 3 (5 - 10hrs) / 4 (>10hrs) for weekly study time |
| failures  | 1-3/4 for the number of class failures (if more than three, then record 4)  |
| schoolsup | yes/no for extra educational support                                        |
| famsup    | yes/no for family educational support                                       |
| paid      | yes/no for extra paid classes for Math or Portuguese                        |
| activities| yes/no for extra-curricular activities                                      |
| nursery   | yes/no for whether attended nursery school                                  |
| higher    | yes/no for the desire to continue studies                                   |
| internet  | yes/no for internet access at home                                          |
| romantic  | yes/no for relationship status                                              |
| famrel    | 1-5 scale on quality of family relationships                                |
| freetime  | 1-5 scale on how much free time after school             |
| goout     | 1-5 scale on how much student goes out with friends      |
| Dalc      | 1-5 scale on how much alcohol consumed on weekdays       |
| Walc      | 1-5 scale on how much alcohol consumed on the weekend    |
| health    | 1-5 scale on health condition                            |
| absences  | 0-93 number of absences from school                      |
| G1        | 0-20 for the first-period grade                          |
| G2        | 0-20 for the second-period grade                         |
| G3        | 0-20 for the final grade                                 |


### Load the data set


Execute the code in the following cell to load your dataset. This code reads the CSV file into a pandas DataFrame, making the data accessible for processing in Python.


In [ ]:
# Student Performance (mathematics course) dataset.
df = pd.read_csv(
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/ZNoKMJ9rssJn-QbJ49kOzA/student-mat.csv"
)

Let's examine the first five rows of the dataset to get a glimpse of the data structure and its contents.


In [ ]:
df.head(5)

You can also review the detailed information for each column in the dataset, focusing on the presence of null values and the specific data types of each column.


In [ ]:
df.info()

## Load LLM


In [ ]:
# Configure the LLM (Groq-hosted Llama). Create a .env file with GROQ_API_KEY=...
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))  # searches up the directory tree for a .env file

from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# Wrap the dataframe in a LangChain agent that can answer questions and plot charts.
agent = create_pandas_dataframe_agent(
    llm,
    df,
    verbose=False,
    return_intermediate_steps=True,   # expose the code the model generates for each chart
    handle_parsing_errors=True,
    allow_dangerous_code=True,        # required: the agent executes the pandas/plot code it writes
    prefix="You are a pandas agent. Always respond using Action/Action Input/Final Answer "
           "format. Never output raw data directly.",
)

### Interact with your data


Let's start with a simple interaction.

Ask LLM how many rows of data are in the CSV file.


In [ ]:
response = agent.invoke("how many rows of data are in this file?")

In [ ]:
response['output']

From the output above, the model reports that there are 395 rows of data in the file.


Let's verify this count using Python code to ensure accuracy.


In [ ]:
len(df)

The row count matches and is correct! 


Curious about the code the LLM generated and used to create this result?

Run the code in the cell below to reveal the underlying commands.


In [ ]:
response['intermediate_steps'][-1][0].tool_input.replace('; ', '\n')

Surprisingly, the LLM uses the same code as you do.


Also, you could let LLM return some data that you are looking for based on the CSV file.


In [ ]:
response = agent.invoke("Give me all the data where student's age is over 18 years old.")

In [ ]:
print(response)

Let's get the code LLM used for charting this plot.


In [ ]:
response['intermediate_steps'][-1][0].tool_input.replace('; ', '\n')

### Plot your data with natural language


#### Task 1
Generating a first visual on the data set to know the total number of male and female students in the data set.

You just need to tell the agent that "Plot the gender count with bars."


In [ ]:
response = agent.invoke("Generate a bar chart to plot the gender count.")

Let's see what code the LLM generated for ploting this chart.


In [ ]:
print(response['intermediate_steps'][-1][0].tool_input.replace('; ', '\n'))

#### Task 2

Generating a pie chart to display the average value of weekend alcohol for each gender in the dataset.

You will use the prompt "Generate a pie chart to display the average value of Walc for each gender."

You may notice that the model generates two charts. The charts indicate the progressive improvement of the agent's code as it searches for the best way to answer your prompt, which improves the response to your query.


In [ ]:
response = agent.invoke("Generate a pie chart to display average value of Walc for each Gender.")

Let's get the code LLM used for charting this plot.


In [ ]:
print(response['intermediate_steps'][-1][0].tool_input.replace('; ', '\n'))

#### Task 3

You can explore the impact of free time on grades based on the data.


In [ ]:
response = agent.invoke("Create box plots to analyze the relationship between 'freetime' (amount of free time) and 'G3' (final grade) across different levels of free time.")

Execute the code below to retrieve the Python script the LLM used for plotting.


In [ ]:
print(response['intermediate_steps'][-1][0].tool_input.replace('; ', '\n'))

#### Task 4

You can explore the effect of alcohol consumption on academic performance.


In [ ]:
response = agent.invoke("Generate scatter plots to examine the correlation between 'Dalc' (daily alcohol consumption) and 'G3', and between 'Walc' (weekend alcohol consumption) and 'G3'.")

Execute the code below to retrieve the Python script the LLM used for plotting.


In [ ]:
print(response['intermediate_steps'][-1][0].tool_input.replace('; ', '\n'))

In [ ]:
response = agent.invoke(
    "Generate scatter plots showing the relationship between "
    "'Medu' (mother's education level) and 'G3' (final grade), "
    "and between 'Fedu' (father's education level) and 'G3'. "
    
)

In [ ]:
response = agent.invoke("Use bar plots to compare the average final grades ('G3') of students with internet access at home versus those without ('internet' column).")

In [ ]:
response = agent.invoke("Plot a scatter plot showing the correlation between the number of absences ('absences') and final grades ('G3') of students.")

for i in range(len(response['intermediate_steps'])):
    print(response['intermediate_steps'][i][0].tool_input.replace(';', '\n'))

## Author

**Anas AlGhannam**  
[github.com/AnasAlghannam](https://github.com/AnasAlghannam)